# HumanForYou - Préparation des données

Maintenant qu'on a une bonne image des données, on les nettoie et on les prépare pour la modélisation. Les modèles ML ne digèrent pas les données brutes telles quelles - il faut fusionner les sources, gérer les valeurs manquantes, encoder les variables texte et normaliser les échelles.

## Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')

RAW  = '../data/raw/'
PROC = '../data/processed/'

## 1. Chargement

On recharge les 5 fichiers sources.

In [2]:
general  = pd.read_csv(RAW + 'general_data.csv')
survey   = pd.read_csv(RAW + 'employee_survey_data.csv')
manager  = pd.read_csv(RAW + 'manager_survey_data.csv')
in_time  = pd.read_csv(RAW + 'in_time.csv',  index_col=0)
out_time = pd.read_csv(RAW + 'out_time.csv', index_col=0)

print(f"general  : {general.shape}")
print(f"survey   : {survey.shape}")
print(f"manager  : {manager.shape}")
print(f"in_time  : {in_time.shape}")
print(f"out_time : {out_time.shape}")

general  : (4410, 24)
survey   : (4410, 4)
manager  : (4410, 3)
in_time  : (4410, 261)
out_time : (4410, 261)


## 2. Suppression des colonnes inutiles

On avait repéré dans l'EDA que 3 colonnes ont la même valeur pour tout le monde. Elles n'apportent aucune information discriminante, on s'en débarrasse.

In [3]:
general = general.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'])
print(f"Dimensions après suppression : {general.shape}")

Dimensions après suppression : (4410, 21)


## 3. Features extraites du badgeage

On a 261 colonnes de timestamps - impossible à donner brut à un modèle. On les condense en 5 indicateurs qui résument le comportement de chaque employé sur l'année :
- `avg_hours_per_day` : heures travaillées en moyenne par jour
- `std_hours_per_day` : variabilité des horaires (quelqu'un avec des horaires très irréguliers...)
- `days_absent` : jours sans badge (absences)
- `days_over_8h` : jours où l'employé a dépassé 8h
- `avg_arrival_hour` : heure d'arrivée moyenne

In [4]:
in_dt  = in_time.apply(pd.to_datetime, errors='coerce')
out_dt = out_time.apply(pd.to_datetime, errors='coerce')
work_hours = (out_dt - in_dt).apply(lambda col: col.dt.total_seconds() / 3600)

badge = pd.DataFrame()
badge['EmployeeID']        = general['EmployeeID'].values
badge['avg_hours_per_day'] = work_hours.mean(axis=1).values
badge['std_hours_per_day'] = work_hours.std(axis=1).values
badge['days_absent']       = work_hours.isna().sum(axis=1).values
badge['days_over_8h']      = (work_hours > 8).sum(axis=1).values
badge['avg_arrival_hour']  = in_dt.apply(
    lambda col: col.dt.hour + col.dt.minute / 60).mean(axis=1).values

print(badge.describe().round(2))

       EmployeeID  avg_hours_per_day  std_hours_per_day  days_absent  \
count     4410.00            4410.00            4410.00      4410.00   
mean      2205.50               7.70               0.30        24.73   
std       1273.20               1.34               0.01         5.50   
min          1.00               5.95               0.25        13.00   
25%       1103.25               6.67               0.29        20.00   
50%       2205.50               7.41               0.30        25.00   
75%       3307.75               8.37               0.31        29.00   
max       4410.00              11.03               0.34        36.00   

       days_over_8h  avg_arrival_hour  
count       4410.00           4410.00  
mean          76.15              9.99  
std          100.02              0.02  
min            0.00              9.93  
25%            0.00              9.98  
50%            6.00              9.99  
75%          205.00             10.00  
max          248.00            

## 4. Fusion des 5 sources

On fusionne tout sur `EmployeeID`.

In [5]:
df = general.merge(survey,  on='EmployeeID', how='left')
df = df.merge(manager, on='EmployeeID', how='left')
df = df.merge(badge,   on='EmployeeID', how='left')

print(f"Dimensions après fusion : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

Dimensions après fusion : (4410, 31)
Colonnes : ['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeID', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating', 'avg_hours_per_day', 'std_hours_per_day', 'days_absent', 'days_over_8h', 'avg_arrival_hour']


## 5. Imputation des valeurs manquantes

On remplace les NA par la médiane de chaque colonne. On a choisi la médiane plutot que la moyenne parce qu'elle est moins sensible aux valeurs extrêmes - un salaire à 1 million ne va pas fausser le remplacement. Et comme on a moins de 1% de NA partout, cette approche est largement suffisante.

In [6]:
print("NA avant imputation :")
print(df.isna().sum()[df.isna().sum() > 0])

for col in df.select_dtypes(include='number').columns:
    if df[col].isna().sum() > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f"  {col} → médiane = {med:.2f}")

print(f"\nNA après imputation : {df.isna().sum().sum()}")

NA avant imputation :
NumCompaniesWorked         19
TotalWorkingYears           9
EnvironmentSatisfaction    25
JobSatisfaction            20
WorkLifeBalance            38
dtype: int64
  NumCompaniesWorked → médiane = 2.00
  TotalWorkingYears → médiane = 10.00
  EnvironmentSatisfaction → médiane = 3.00
  JobSatisfaction → médiane = 3.00
  WorkLifeBalance → médiane = 3.00

NA après imputation : 0


## 6. Encodage des variables texte

Les modèles ML ne comprennent que des nombres. Il faut tout convertir.

- `Attrition` : on code Yes/No en 1/0, c'est notre variable cible
- `Gender` : binaire, un LabelEncoder suffit (0/1)
- Les autres catégorielles (`BusinessTravel`, `Department`, etc.) : One-Hot Encoding

Pourquoi One-Hot plutot que des entiers ? Parce que si on met 0/1/2 pour Marié/Célibataire/Divorcé, le modèle va croire que Divorcé > Marié. C'est n'importe quoi pour une variable nominale.

In [7]:
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)
print("Attrition :", df['Attrition'].value_counts().to_dict())

le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
print(f"Gender : {dict(zip(le.classes_, le.transform(le.classes_)))}")

ohe_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print(f"\nDimensions après encodage : {df.shape}")

Attrition : {0: 3699, 1: 711}
Gender : {'Female': np.int64(0), 'Male': np.int64(1)}

Dimensions après encodage : (4410, 45)


## 7. Normalisation

`StandardScaler` ramène chaque variable numérique à moyenne=0 et écart-type=1. C'est indispensable pour les modèles sensibles aux échelles comme la régression logistique, le k-NN ou le SVM - sinon le salaire (en milliers) va complètement écraser le niveau d'éducation (1 à 5).

On ne normalise pas la cible (`Attrition`), l'identifiant (`EmployeeID`) ni les colonnes One-Hot (déjà en 0/1).

In [8]:
cols_exclure = ['EmployeeID', 'Attrition']
ohe_generated = [c for c in df.columns if any(c.startswith(b + '_') for b in
                 ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus'])]
cols_exclure += ohe_generated

cols_norm = [c for c in df.select_dtypes(include='number').columns if c not in cols_exclure]

scaler = StandardScaler()
df[cols_norm] = scaler.fit_transform(df[cols_norm])

print("Colonnes normalisées :", cols_norm)
print()
df[cols_norm[:4]].describe().round(2)

Colonnes normalisées : ['Age', 'DistanceFromHome', 'Education', 'Gender', 'JobLevel', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating', 'avg_hours_per_day', 'std_hours_per_day', 'days_absent', 'days_over_8h', 'avg_arrival_hour']



,Age,DistanceFromHome,Education,Gender
count,4410.00,4410.00,4410.00,4410.00
mean,-0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00
min,-2.07,-1.01,-1.87,-1.22
25%,-0.76,-0.89,-0.89,-1.22
50%,-0.10,-0.27,0.09,0.82
75%,0.67,0.59,1.06,0.82
max,2.53,2.44,2.04,0.82


## 8. Vérification et sauvegarde

In [9]:
print(f"Shape final  : {df.shape}")
print(f"NA restants  : {df.isna().sum().sum()}")
print(f"Attrition    : {df['Attrition'].value_counts().to_dict()}")

df.head(3)

Shape final  : (4410, 45)
NA restants  : 0
Attrition    : {0: 3699, 1: 711}


,Age,Attrition,DistanceFromHome,Education,EmployeeID,Gender,JobLevel,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,...,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,1.541369,0,-0.393938,-0.891688,1,-1.224745,-0.961486,1.405136,-0.678464,-1.150554,...,False,False,False,False,False,False,False,False,True,False
1,-0.648668,1,0.099639,-1.868426,2,-1.224745,-0.961486,-0.491661,-1.079486,2.129306,...,False,False,False,False,False,True,False,False,False,True
2,-0.539166,0,0.963398,1.061787,3,0.816497,1.749610,2.725053,-0.678464,-0.057267,...,False,False,False,False,False,False,True,False,True,False


In [10]:
df.to_csv(PROC + 'dataset_final.csv', index=False)
print(f"Sauvegardé → {PROC}dataset_final.csv  ({df.shape})")

Sauvegardé → ../data/processed/dataset_final.csv  ((4410, 45))
